# **Análisis de Movilidad - TfL London Stations**
Este proyecto analiza el flujo de pasajeros en la red de transporte de Londres en los años 2012, 2017 y 2021. A continuación, se detalla el diccionario de datos:

| Campo | Descripción |
| :--- | :---: |
| NLC | Código Nacional de Ubicación de la estación |
| Station | El nombre de la estación |
| En/Ex 20xx | El número total de entradas y salidas de una estación determinada en el año, desde 2007 hasta 2021 |
| LINES | Indica todas las líneas de TfL que operan en una estación determinada |
| NETWORK | Indica si la estación forma parte del metro de Londres |
| London Underground / Elizabeth Line / London Overground / DLR | Indica si esta estación forma parte de la red indicada |
| Night Tube? | Indica si una estación es parte del servicio nocturno de Tube y London Overground que opera los viernes y sábados por la noche |

# Parte 1

## Reconocimiento, Tratado y Limpieza de Nulls

In [196]:
import pandas as pd

tflStationData = pd.read_csv('TfL_stations.csv')

tflStationData.dropna(how='all', inplace=True)

porcentaje_de_nulls = tflStationData.isna().sum().sum()*100/tflStationData.size

print(porcentaje_de_nulls)

29.522935779816514


El cálculo anterior refiere al porcentaje de nulls que cuenta el archivo original y debemos tratar. Para ellos realizaremos un data cleaning y eventualmente, el resultado, lo llevare a una Tercer Forma Normal (3FN) que nos permitirá el trabajo correcto de análisis.

Para poder convertir estos nulls en un dato consistente a su columna se podrán tomar dos caminos que llevarán al mismo lugar, aunque realizando efectivamente el segundo para el DataFrame.

Los enfoques son:
1. Tomar los tipos de datos de cada columna, para luego iterar sobre ellas en el DataFrame y, ante cada null, reemplazar por un valor consistente al dato correspondiente. Así desarrollo una lógica manual para entender la distribución de tipos de datos y aplicar una limpieza controlada.

In [197]:
tipos_de_datos = []

for type in tflStationData.dtypes:
    tipo = type.name
    if (tipo not in tipos_de_datos):
        tipos_de_datos.append(tipo)


df_limpio_manual = tflStationData.copy() #Se realiza una copia para no modificar el DataFrame original y poder enseñar el camino 1 a modo de ejemplo

#Ahora es la corrección de forma manual
for columna in df_limpio_manual:

    if (df_limpio_manual[columna].dtype.name in ['int64', 'float64']):
        df_limpio_manual[columna] = df_limpio_manual[columna].fillna(-1)
    else:
        df_limpio_manual[columna] = df_limpio_manual[columna].fillna('Sin texto')

porcentaje_de_nulls_manual = df_limpio_manual.isna().sum().sum()*100/df_limpio_manual.size

print(porcentaje_de_nulls_manual) #Output = 0.0, significa que se ha cumplido el objetivo ya que no hay nulls en el DataFrame df_limpio_manual

0.0


2. Uso de metodo nativos de Pandas (select_dtypes) para priorizar un código limpio, breve, legible y optimizado. 

In [198]:
lista_columnas_numericas = tflStationData.select_dtypes(include=['number']).columns

tflStationData[lista_columnas_numericas] = tflStationData[lista_columnas_numericas].fillna(-1)

lista_columnas_no_numericas = tflStationData.select_dtypes(exclude=['number']).columns

tflStationData[lista_columnas_no_numericas] = tflStationData[lista_columnas_no_numericas].fillna('Sin texto')

porcentaje_de_nulls_optimizado = tflStationData.isna().sum().sum()*100/tflStationData.size

print(porcentaje_de_nulls_optimizado) #Output = 0.0, completando la limpieza de nulls en nuestro DataFrame basandonos en metodos nativos y optimizados de Pandas

0.0


# Parte 2

## Reestructuración, Limpieza Final y Normalización

En esta parte, el objetivo es la reestructuración del DataFrame para que se encuentre en una Tercer Forma Normal (3FN), y así poder realizar nuestras consultas y conclusiones luego en el análisis.

Al realizar una vista al archivo TfL_stations.csv se nota que hay una estructura ineficiente: si se desean agregar datos al paso de los años implica añadir columnas, cuando preferentemente se desea añadir filas; columnas como "LINES" tiene múltiples valores separados por comas, no llegando a una atomicidad; datos que deberían ser de tipo Int pero resultan ser Float; y dos columnas (NETWORK y London Underground) que nos dan la misma información, siendo redundante.

En el siguiente código se encontrará la fundamentación de las observaciones previas y consultas de verificación (como por ejemplo, si hay filas duplicadas, entre otras consultas).

- 1) Secuencia de valores separados por comas en LINES. Para ello, la solución será atomizar cada conjunto de valores.

In [199]:
print(tflStationData['LINES'].str.contains(',').sum()) #Output = 102, corroborando que tenemos una lista de valores en 102 filas

tflStationData['LINES'] = tflStationData['LINES'].str.split(',').explode('LINES')

print(tflStationData['LINES'].str.contains(',').sum()) #Output = 0, ya no tenemos listas en LINES aunque sí filas por optimizar ya que multiplicamos su contenido en el archivo al separar los valores de dicha columna

102
0


- 2) Filas duplicadas. Se corrobora la no existencia de filas duplicadas.

In [200]:
print(tflStationData.duplicated().sum()) #Output = 0, lo cual significa que hay 0 filas que son copias exactas de otra

0


- 3) Tipos de datos distintos en mismas columnas númericas (Int vs Float). La solución será unificar ello en columnas de ints (ya que será consistente con el dato de ingresos y egresos de personas).

In [201]:
print(tflStationData.dtypes) #El output enseñará que las columnas númericas estan dadas por float64

Unnamed: 0              int64
NLC                   float64
Station                object
En/Ex 2007            float64
En/Ex 2008            float64
En/Ex 2009            float64
En/Ex 2010            float64
En/Ex 2011            float64
En/Ex 2012            float64
En/Ex 2013            float64
En/Ex 2014            float64
En/Ex 2015            float64
En/Ex 2016            float64
En/Ex 2017            float64
En/Ex 2018            float64
En/Ex 2019            float64
En/Ex 2020            float64
En/Ex 2021            float64
LINES                  object
NETWORK                object
London Underground     object
Elizabeth Line         object
London Overground      object
DLR                    object
Night Tube?            object
dtype: object


In [202]:
columnas_mumericas = tflStationData.select_dtypes(include=['number']).columns.to_list()

tflStationData[columnas_mumericas] = tflStationData[columnas_mumericas].astype(int) #Casteamos al tipo de dato deseado

print(tflStationData.dtypes) #El output enseña que se ha hecho correctamente el casteo y trabajaremos con datos congruentes al contexto (números sin decimales)

Unnamed: 0             int64
NLC                    int64
Station               object
En/Ex 2007             int64
En/Ex 2008             int64
En/Ex 2009             int64
En/Ex 2010             int64
En/Ex 2011             int64
En/Ex 2012             int64
En/Ex 2013             int64
En/Ex 2014             int64
En/Ex 2015             int64
En/Ex 2016             int64
En/Ex 2017             int64
En/Ex 2018             int64
En/Ex 2019             int64
En/Ex 2020             int64
En/Ex 2021             int64
LINES                 object
NETWORK               object
London Underground    object
Elizabeth Line        object
London Overground     object
DLR                   object
Night Tube?           object
dtype: object


- 4) Columnas iguales, información redundante. La columna NETWORK indica si la estación forma parte del metro de Londres, y en caso de ser así escribe "London Underground"; y la columna "London Underground" indica si la estación forma parte de dicha red, y en caso de ser así escribe "Yes". Motivo por el cual trasmite la misma información, pero para demostrarlo se debe de cambiar el resultado de una columna para unificar sus valores respecto a la otra. Por ello, se reemplazará en la columna NETWORK las palabras "London Underground" por "Yes". Y luego se realizará la consulta correspondiente para verificar si ambas columnas son exactamente iguales, dando el poder así de eliminar una de esas dos columnas (NETWORK).

In [203]:
tflStationData['NETWORK'] = tflStationData['NETWORK'].replace('London Underground', 'Yes')

print(tflStationData['NETWORK'].equals(tflStationData['London Underground'])) #Output = True, por lo que procedemos con la eliminacion de la columna NETWORK

tflStationData = tflStationData.drop(columns=['NETWORK'])

True


Dado que ya se han realizado las limpiezas necesarias se pasará ahora a la reestructuración del DataFrame antes de conseguir que llegue a la 3FN. Para ello, el punto crítico más importante son las columnas que representan a los años y generan una ineficiencia en el guardado de datos. Entonces se añadirán dos columnas nuevas, "Año" y "Flujo_pasajeros" y se eliminarán todas aquellas columnas que representaban los ingresos y egresos por año (aunque manteniendo dicha información a modo de filas), para que añadir más datos implique un crecimiento de filas y no de columnas, dando una reorganización más útil y óptima.

In [204]:
df_a_modificar = tflStationData.filter(like='En/Ex') #Se toma un nuevo DataFrame que contiene las columnas que mencionan los años

columnas_fijas = tflStationData.columns[~tflStationData.columns.str.contains('En/Ex')]

df_a_mantener = tflStationData[columnas_fijas] #Se toma un nuevo DataFrame con las columnas que no contienen los años y quedarán como antes

df_final = tflStationData.melt(columnas_fijas.to_list(), df_a_modificar.columns.to_list(), 'Año', 'Flujo_Pasajeros') #Se trabajará con el DataFrame df_final que ya contiene las columnas según la forma deseada

Dada la nueva estructura, ahora se retocará el DataFrame de la siguiente forma para mayor prolijidad:

In [205]:
df_final['Año'] = df_final['Año'].str.replace(r'.*En/Ex\s*(.*)', r'\1', regex = True) #En la columna "Año" se pasará de tener "En/Ex <año>" a solamente el año

df_final['Año'] = df_final['Año'].astype(int) #Casteamos la columna a int, ya que ahora solo tenemos números enteros y no strings

print(df_final.columns) #Se nota que aparece una columna no deseada titulada "Unnamed: 0"

df_final = df_final.drop(columns=['Unnamed: 0']) #Quitamos la columna que no aportaba información al DataFrame

df_final[['London Underground', 'Elizabeth Line', 'London Overground', 'DLR']] = df_final[['London Underground', 'Elizabeth Line', 'London Overground', 'DLR']].replace('Sin texto', 'No') #Donde antes habían nulls y la preocupación era quitarlos, ahora será ajustarlos al contenido de la columna, colocando un "No". 
#En cambio, La columna "Night Tube?" tiene "Yes" y "No", pero también nulls, por lo que ellos si quedarán como "Sin texto"

df_final.dtypes #Con esto se corrobora que se ha realizado el casteo y eliminado la columna correctamente

Index(['Unnamed: 0', 'NLC', 'Station', 'LINES', 'London Underground',
       'Elizabeth Line', 'London Overground', 'DLR', 'Night Tube?', 'Año',
       'Flujo_Pasajeros'],
      dtype='object')


NLC                    int64
Station               object
LINES                 object
London Underground    object
Elizabeth Line        object
London Overground     object
DLR                   object
Night Tube?           object
Año                    int64
Flujo_Pasajeros        int64
dtype: object

Se desea un orden por NLC ascendente, por ello se corrobora su estado previo y se modifica ya que no se hallaba de la forma deseada.

In [206]:
print(df_final['NLC'].equals(df_final['NLC'].sort_values(ascending= True))) #Output = False

df_final = df_final.sort_values(by = 'NLC', ascending = True)

print(df_final['NLC'].equals(df_final['NLC'].sort_values(ascending= True))) #Output = True

False
True


Resumidamente, se ha logrado una estructura que permitirá trabajar cómodamente luego. Para entender sobre donde se está trabajando, se debe saber que la estructura actual es la siguiente:

df_final(NLC, Station, LINES, London Underground, Elizabeth Line, London Overground, DLR, Night Tube?, Año, Flujo_Pasajeros)

La idea actual es llegar a la 3FN, que para ello se debe primero ver que cumpla 1FN y 2FN:

- 1FN &rarr; cumple ya que todas las columnas (inclusive LINES que fue tratada específicamente para corregir su estructura) tiene solamente valores atómicos, no hay grupos repetidos (fue eliminada la columna NETWORK que repetía información) y contamos con una clave primaria (en este caso, compuesta por NLC y Año).
- 2FN &rarr; cumple 1FN, pero existen dependencias funcionales parciales; significa que hay atributos que dependen de parte de la clave (como Station con NLC) y no de su totalidad (del Año). Por lo que se debe dividir el DataFrame actual en 
    - df_estaciones(NLC, Station, LINES, London Underground, Elizabeth Line, London Overground, DLR, Night Tube?)
    - df_flujo(NLC, Año, Flujo_Pasajeros)

In [207]:
df_estaciones = df_final[['NLC', 'Station', 'LINES', 'London Underground', 'Elizabeth Line', 'London Overground', 'DLR', 'Night Tube?']]

df_estaciones = df_estaciones.rename(columns = {'Station' : 'Estacion', 'LINES' : 'Lineas', 'Night Tube?' : 'Metro_nocturno?'})

df_flujo = df_final[['NLC', 'Año', 'Flujo_Pasajeros']]

Es importante decir que el melt hecho para reestructurar el DataFrame ha multiplicado filas. Por lo que, antes de seguir analizando Formas Normales, se realizará la consulta de existencia de duplicados y (por ser cierto que existen) la limpieza de los mismos.

In [208]:
print(df_estaciones.duplicated().sum())

df_estaciones = df_estaciones.drop_duplicates()

print(df_estaciones.duplicated().sum())

6104
0


In [209]:
print(df_flujo.duplicated().sum())

df_flujo = df_flujo.drop_duplicates()

print(df_flujo.duplicated().sum())

15
0


- 3FN &rarr; cumple 2FN y no hay dependencias transitivas entre atributos.

Por lo que los DataFrames, df_estaciones y df_flujo, ya están listos para ser analizados.

# Parte 3

## Ingeniería de Datos en Archivos Complementarios para un Análisis Integral

En esta parte el interés se encuentra en que los DataFrames pasen a ser archivos en formatos útiles para analizar bajo consultas SQL, también en unificar la estructura de los archivos que nos detallan los movimientos de ingresos y egresos en los años 2012, 2017 y 2021, los cuales serán el punto de conexión para análisis luego.

Lo importante será la limpieza y equiparar la estructura de los 3 archivos que representan cada año para que el trabajo en SQL nos permita flexibilidad y una óptima interacción en consultas.

Además de lo dicho, se obtendrá un archivo .csv para cada DataFrame previamente trabajado.

In [210]:
df_estaciones.to_csv('estaciones_limpias.csv', index = False, encoding = 'utf-8')

df_flujo.to_csv('flujo_pasajeros.csv', index = False, encoding = 'utf-8')

Dado esto, como se realizarán los análisis de los años siguientes, para cada archivo se desarrollará una sección de código (donde se hallará el proceso de ingeniería de datos) junto a al año y el motivo de selección:

(Es importante aclarar que no todos los archivos comparten la misma estructura, por lo que se trabajará para unificar su formato y que las consultas de análisis mantengan prolijidad y facilidad en lectura)

- 2012 &rarr; ya que Londres fue sede de los Juegos Olímpicos de dicho año.

In [211]:
df_olimpics = pd.read_csv('Weekly Data 2012.csv', skiprows=6) #Se saltea las primeras 6 líneas que no aportan información

df_olimpics.dropna(how='all', inplace = True)

print(df_olimpics.columns.to_list())

df_olimpics.columns = df_olimpics.columns.str.strip().str.replace('\n', ' ').str.lower() #Con el .strip() eliminamos los vacios dentro de cada nombre de columna, dejando únicamente los strings que no son vacíos; con el .str.replace() reemplazamos los saltos de línea por espacios y con el .lower() hacemos que los nombres de las columnas queden en mínuscula, que nos es útil para luego trabajar bajo un criterio único

print(df_olimpics.columns.to_list())

df_olimpics = df_olimpics.drop(columns=['note']) #Se decide eliminar esta columna ya que no aportaba información para las próximas consultas

nuevas_columnas_2012 = {'nlc' : 'NLC', 
                   'station' : 'Estacion', 
                   'weekday' : 'Entradas_semana', 
                   'saturday' : 'Entradas_sabado', 
                   'sunday' : 'Entradas_domingo',
                   'weekday.1' : 'Salidas_semana', 
                   'saturday.1' : 'Salidas_sabado', 
                   'sunday.1' : 'Salidas_domingo', 
                   'million' : 'Entradas_salidas_anual_millones'}

df_olimpics = df_olimpics.rename(columns = nuevas_columnas_2012)

'''Se verifica que se cuenta con un DataFrame limpio'''

print(df_olimpics.isna().sum().sum()) #Output = 0, no hay nulls en el DataFrame

print(df_olimpics.duplicated().sum()) #Output = 0, no existen filas duplicadas

df_olimpics.head()

['nlc', 'Station', 'Note', 'Weekday', 'Saturday', 'Sunday', 'Weekday.1', 'Saturday.1', 'Sunday.1', 'million']
['nlc', 'station', 'note', 'weekday', 'saturday', 'sunday', 'weekday.1', 'saturday.1', 'sunday.1', 'million']
0
0


,NLC,Estacion,Entradas_semana,Entradas_sabado,Entradas_domingo,Salidas_semana,Salidas_sabado,Salidas_domingo,Entradas_salidas_anual_millones
0,500.0,Acton Town,8905.0,6408.0,4360.0,8619.0,5960.0,4254.0,5.58
1,502.0,Aldgate,12031.0,2384.0,1946.0,11942.0,3471.0,2760.0,6.65
2,503.0,Aldgate East,16210.0,11047.0,10285.0,15070.0,9593.0,9154.0,10.13
3,505.0,Alperton,4450.0,3193.0,2264.0,4612.0,3189.0,2290.0,2.89
4,506.0,Amersham,3387.0,1675.0,1133.0,3708.0,1703.0,1024.0,2.10


In [212]:
'''Se ordena por NLC descendente, se elimina la primer fila que está vacía y se reestructura el índice'''

print(df_olimpics['NLC'].equals(df_olimpics['NLC'].sort_values(ascending= True))) #Output = False

df_olimpics = df_olimpics.sort_values(by = 'NLC', ascending = True)

print(df_olimpics['NLC'].equals(df_olimpics['NLC'].sort_values(ascending= True))) #Output = True

df_olimpics = df_olimpics.drop(df_olimpics.index[0])

df_olimpics = df_olimpics.reset_index(drop = True)

False
True


In [213]:
print(df_olimpics.dtypes) #Se verifican los tipos de datos y, debido a que no se encuentran de la forma deseada, se corrigen

columnas_a_castear_2012 = df_olimpics.drop(columns=['Estacion', 'Entradas_salidas_anual_millones']).columns.to_list()

df_olimpics[columnas_a_castear_2012] = df_olimpics[columnas_a_castear_2012].astype(int)

print(df_olimpics.dtypes) #Verificación de corrección de datos

NLC                                float64
Estacion                            object
Entradas_semana                    float64
Entradas_sabado                    float64
Entradas_domingo                   float64
Salidas_semana                     float64
Salidas_sabado                     float64
Salidas_domingo                    float64
Entradas_salidas_anual_millones    float64
dtype: object
NLC                                  int64
Estacion                            object
Entradas_semana                      int64
Entradas_sabado                      int64
Entradas_domingo                     int64
Salidas_semana                       int64
Salidas_sabado                       int64
Salidas_domingo                      int64
Entradas_salidas_anual_millones    float64
dtype: object


In [214]:
df_olimpics.head() #Se visualiza el resultado del código previo

,NLC,Estacion,Entradas_semana,Entradas_sabado,Entradas_domingo,Salidas_semana,Salidas_sabado,Salidas_domingo,Entradas_salidas_anual_millones
0,501,Barbican,17136,6241,4267,17189,6432,4332,9.85
1,502,Aldgate,12031,2384,1946,11942,3471,2760,6.65
2,503,Aldgate East,16210,11047,10285,15070,9593,9154,10.13
3,505,Alperton,4450,3193,2264,4612,3189,2290,2.89
4,506,Amersham,3387,1675,1133,3708,1703,1024,2.10


In [215]:
df_olimpics.to_csv('movimientos_año_2012.csv', index = False, encoding = 'utf-8')

- 2017 &rarr; ya que en el año 2016 se incorporó el servicio nocturno (Night Tube) y se desea comprender como afectó al movimiento viajes en las redes de metro una vez consolidado este sistema.

In [216]:
df_nocturno = pd.read_csv('Weekly Data 2017.csv', skiprows=6)

df_nocturno.dropna(how='all', inplace=True)

df_nocturno.columns = df_nocturno.columns.str.strip().str.replace('\n', ' ').str.lower()

print(df_nocturno.columns.to_list())

df_nocturno = df_nocturno.drop(columns = ['note'])

nuevas_columnas = {'nlc' : 'NLC', 
                   'station' : 'Estacion',
                   'borough' : 'Ciudad', 
                   'weekday' : 'Entradas_semana', 
                   'saturday' : 'Entradas_sabado', 
                   'sunday' : 'Entradas_domingo', 
                   'weekday.1' : 'Salidas_semana', 
                   'saturday.1' : 'Salidas_sabado', 
                   'sunday.1' : 'Salidas_domingo', 
                   'million' : 'Entradas_salidas_anual_millones'}

df_nocturno = df_nocturno.rename(columns = nuevas_columnas)

df_nocturno = df_nocturno.drop(columns = ['Ciudad'])

print(df_nocturno.columns.to_list())

'''Se verifica que se cuenta con un DataFrame limpio'''

print(df_nocturno.isna().sum().sum()) #Output = 0, no hay nulls en el DataFrame

print(df_nocturno.duplicated().sum()) #Output = 0, no existen filas duplicadas

df_nocturno.head()

['nlc', 'station', 'borough', 'note', 'weekday', 'saturday', 'sunday', 'weekday.1', 'saturday.1', 'sunday.1', 'million']
['NLC', 'Estacion', 'Entradas_semana', 'Entradas_sabado', 'Entradas_domingo', 'Salidas_semana', 'Salidas_sabado', 'Salidas_domingo', 'Entradas_salidas_anual_millones']
0
0


,NLC,Estacion,Entradas_semana,Entradas_sabado,Entradas_domingo,Salidas_semana,Salidas_sabado,Salidas_domingo,Entradas_salidas_anual_millones
0,625,King's Cross St. Pancras,149150,117237,98579,146864,112360,89348,97.92
1,747,Waterloo,147574,91841,55080,149577,90703,56770,91.27
2,669,Oxford Circus,121364,103026,63185,136405,113894,65547,84.09
3,741,Victoria,121018,88605,69760,122783,91001,71565,79.36
4,635,London Bridge,113606,78569,45408,107048,75800,42694,69.05


In [217]:
'''Se ordena por NLC descendente y se reestructura el índice'''

print(df_nocturno['NLC'].equals(df_nocturno['NLC'].sort_values(ascending= True))) #Output = False

df_nocturno = df_nocturno.sort_values(by = 'NLC', ascending = True)

print(df_nocturno['NLC'].equals(df_nocturno['NLC'].sort_values(ascending= True))) #Output = True

df_nocturno = df_nocturno.reset_index(drop=True)

print(df_nocturno.dtypes)

False
True
NLC                                  int64
Estacion                            object
Entradas_semana                      int64
Entradas_sabado                      int64
Entradas_domingo                     int64
Salidas_semana                       int64
Salidas_sabado                       int64
Salidas_domingo                      int64
Entradas_salidas_anual_millones    float64
dtype: object


In [218]:
df_nocturno.head()

,NLC,Estacion,Entradas_semana,Entradas_sabado,Entradas_domingo,Salidas_semana,Salidas_sabado,Salidas_domingo,Entradas_salidas_anual_millones
0,500,Acton Town,9531,6716,4744,9382,6617,4785,6.04
1,501,Barbican,20680,7416,5149,20512,7702,5351,11.83
2,502,Aldgate,15080,4397,3261,16023,5909,4230,8.85
3,503,Aldgate East,22327,16166,13323,21071,13893,11347,14.00
4,505,Alperton,4495,3279,2345,5081,3392,2445,3.05


In [219]:
df_nocturno.to_csv('movimientos_año_2017.csv', index = False, encoding = 'utf-8')

- 2021 &rarr; un año pasada la pandemia del 2020 que permite entender como se transformó la ciudad y la población adaptó su movimiento a sus nuevos marcos de funcionamiento social.

In [220]:
df_pandemia = pd.read_csv('AC2021_AnnualisedEntryExit - Annualised.csv', skiprows = 6)

df_pandemia.dropna(how = 'all', inplace = True)

df_pandemia = df_pandemia.drop(columns = ['Mode', 'ASC', 'Coverage', 'Source'])

df_pandemia['Entradas_semana'] = df_pandemia['Entries'] + df_pandemia['Entries.1']

df_pandemia['Salidas_semana'] = df_pandemia['Exits'] + df_pandemia['Exits.1']

nuevas_columnas_2017 = {'Entries.2' : 'Entradas_sabado',
                        'Entries.3' : 'Entradas_domingo',
                        'Exits.2' : 'Salidas_sabado',
                        'Exits.3' : 'Salidas_domingo',
                        'En/Ex' : 'Entradas_salidas_anual_millones',
                        'Station' : 'Estacion'}

df_pandemia = df_pandemia.rename(columns = nuevas_columnas_2017)

df_pandemia = df_pandemia.drop(columns = ['Entries', 'Entries.1', 'Exits', 'Exits.1'])

df_pandemia = df_pandemia[['NLC', 'Estacion', 'Entradas_semana', 'Entradas_sabado', 'Entradas_domingo', 'Salidas_semana', 'Salidas_sabado', 'Salidas_domingo', 'Entradas_salidas_anual_millones']]

print(df_pandemia.columns.to_list())

'''Se verifica que se cuenta con un DataFrame limpio'''

print(df_pandemia.isna().sum().sum()) #Output = 0, no hay nulls en el DataFrame

print(df_pandemia.duplicated().sum()) #Output = 0, no existen filas duplicadas

df_pandemia.head()

['NLC', 'Estacion', 'Entradas_semana', 'Entradas_sabado', 'Entradas_domingo', 'Salidas_semana', 'Salidas_sabado', 'Salidas_domingo', 'Entradas_salidas_anual_millones']
0
0


,NLC,Estacion,Entradas_semana,Entradas_sabado,Entradas_domingo,Salidas_semana,Salidas_sabado,Salidas_domingo,Entradas_salidas_anual_millones
0,500,Acton Town,"6,8516,899","5,657","3,961","6,8866,691","5,559","4,174","2,902,697"
1,502,Aldgate,"8,0386,690","5,035","3,585","9,6558,611","7,453","4,595","3,525,128"
2,503,Aldgate East,"12,75113,270","13,617","9,980","11,98412,882","12,951","8,261","5,611,130"
3,505,Alperton,"3,1093,121","2,543","1,654","3,3473,325","2,537","1,732","1,345,253"
4,506,Amersham,"2,3842,249","1,745","1,089","2,3152,175","1,545","1,063","946,577"


In [221]:
'''Se ordena por NLC descendente, se elimina la primer fila que está vacía y se reestructura el índice'''

print(df_pandemia['NLC'].equals(df_pandemia['NLC'].sort_values(ascending= True))) #Output = False

df_pandemia = df_pandemia.sort_values(by = 'NLC', ascending = True)

print(df_pandemia['NLC'].equals(df_pandemia['NLC'].sort_values(ascending= True))) #Output = True

df_pandemia = df_pandemia.reset_index(drop = True)

columnas_a_corregir_2021 = df_pandemia.drop(columns=['Estacion', 'NLC']).columns.to_list()

for columna in columnas_a_corregir_2021:

    if columna != 'Entradas_salidas_anual_millones':
        df_pandemia[columna] = df_pandemia[columna].astype(str).str.replace(',', '')

        df_pandemia[columna] = df_pandemia[columna].astype(int)
    else:
        df_pandemia[columna] = (df_pandemia[columna].astype(str).str.replace(',', '').astype(float)/1000000).round(2)

print(df_pandemia.dtypes)

False
True
NLC                                  int64
Estacion                            object
Entradas_semana                      int64
Entradas_sabado                      int64
Entradas_domingo                     int64
Salidas_semana                       int64
Salidas_sabado                       int64
Salidas_domingo                      int64
Entradas_salidas_anual_millones    float64
dtype: object


In [222]:
df_pandemia.head()

,NLC,Estacion,Entradas_semana,Entradas_sabado,Entradas_domingo,Salidas_semana,Salidas_sabado,Salidas_domingo,Entradas_salidas_anual_millones
0,500,Acton Town,68516899,5657,3961,68866691,5559,4174,2.90
1,501,Barbican,87787849,4749,3030,93778407,5183,3168,3.47
2,502,Aldgate,80386690,5035,3585,96558611,7453,4595,3.53
3,503,Aldgate East,1275113270,13617,9980,1198412882,12951,8261,5.61
4,504,Stratford International DLR,56095856,5079,3662,48105257,4600,3488,2.62


In [223]:
df_pandemia.to_csv('movimientos_año_2021.csv', index = False, encoding = 'utf-8')